Block 0 — Install packages

In [ ]:
!pip install -q datasets anthropic scikit-learn tqdm pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.2/388.2 kB 6.1 MB/s eta 0:00:00


### Block 1 — Load FPB dataset from Hugging Face

In [ ]:
from datasets import load_dataset

# Change this to your actual FPB dataset id if different
DATASET_NAME = "ChanceFocus/en-fpb"
# If your split is not "test", change here (e.g. "train")
dataset = load_dataset(DATASET_NAME, split="test")

print(dataset)
print("Number of examples:", len(dataset))
print("Example 0:", dataset[0])


README.md:   0%|          | 0.00/742 [00:00<?, ?B/s]

data/train-00000-of-00001-ab9a3b4799b095(…):   0%|          | 0.00/608k [00:00<?, ?B/s]

data/test-00000-of-00001-8bd1e21c671fb67(…):   0%|          | 0.00/188k [00:00<?, ?B/s]

data/valid-00000-of-00001-303e4ba2afe838(…):   0%|          | 0.00/154k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/970 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/776 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
    num_rows: 970
})
Number of examples: 970
Example 0: {'id': 'fpb3876', 'query': 'Analyze the sentiment of this statement extracted from a financial news article. Provide your answer as either negative, positive, or neutral.\nText: The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .\nAnswer:', 'answer': 'positive', 'text': 'The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .', 'choices': ['positive', 'neutral', 'negative'], 'gold': 0}


### Block 2 — Label config + prediction normalization

In [ ]:

import re
from typing import List, Optional

from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
import pandas as pd

LABELS = ["negative", "neutral", "positive"]

def normalize_prediction(raw: Optional[str]) -> str:
    """
    Map a free-form model output to one of the three sentiment labels.
    """
    if raw is None:
        return "neutral"
    text = raw.strip().lower()

    # Try exact match
    for label in LABELS:
        if text == label:
            return label

    # Try substring / word match
    for label in LABELS:
        if re.search(r"\b" + re.escape(label) + r"\b", text):
            return label

    # Fallback
    return "neutral"


### Block 3 — Claude Sonnet 4 client + single-sentence classifier

In [ ]:
import os
import time
import getpass
from anthropic import Anthropic

# Ensure Anthropic API key is set
if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key (sk-...): ")

client_claude = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

MODEL_ID_CLAUDE = "claude-sonnet-4-20250514"

SYSTEM_PROMPT_CLAUDE = (
    "You are a financial sentiment classifier.\n"
    "Given a single sentence from a financial news article or microblog,\n"
    "classify its overall sentiment from the perspective of an investor as\n"
    "exactly one of: negative, neutral, or positive.\n"
    "Respond with only one word: negative, neutral, or positive."
)

def classify_with_claude_sonnet4(sentence: str,
                                 max_retries: int = 6,
                                 sleep_base: float = 1.0) -> str:
    """
    Call Claude Sonnet 4 once and return a normalized three-way label.
    """
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client_claude.messages.create(
                model=MODEL_ID_CLAUDE,
                max_tokens=16,
                temperature=0.0,
                system=SYSTEM_PROMPT_CLAUDE,
                messages=[{"role": "user", "content": sentence}],
            )
            # Anthropic messages API: text blocks live in resp.content
            parts = []
            for block in resp.content:
                if getattr(block, "type", None) == "text":
                    parts.append(getattr(block, "text", ""))
            raw = "".join(parts).strip() if parts else ""
            return normalize_prediction(raw)
        except Exception as e:
            last_err = e
            wait = sleep_base * (2 ** attempt)
            print(f"[Claude Sonnet 4] Error on attempt {attempt+1}/{max_retries}: {last_err}")
            time.sleep(wait)

    print("[Claude Sonnet 4] Failed after retries, returning 'neutral'.")
    return "neutral"


Anthropic API key (sk-...): ··········


In [7]:
y_true = []
y_pred = []

print(f"Evaluating model: {MODEL_ID_CLAUDE} on {len(dataset)} FPB examples...\n")

for example in tqdm(dataset, total=len(dataset)):
    sentence = example["text"]
    true_label = example["answer"].lower().strip()
    pred_label = classify_with_claude_sonnet4(sentence)

    y_true.append(true_label)
    y_pred.append(pred_label)

print("Total examples:", len(dataset))
print("Got predictions for:", len(y_pred))


Evaluating model: claude-sonnet-4-20250514 on 970 FPB examples...



100%|██████████| 970/970 [36:00<00:00,  2.23s/it]

Total examples: 970
Got predictions for: 970


### Block 4 — Evaluation loop on FPB

In [8]:
# Overall metrics
accuracy = accuracy_score(y_true, y_pred)
print(f"\nOverall accuracy: {accuracy:.4f}\n")

print("Classification report:")
print(classification_report(y_true, y_pred, labels=LABELS))

# Put everything in a DataFrame for inspection
df = pd.DataFrame({
    "text": [ex["text"] for ex in dataset],
    "gold": y_true,
    "pred": y_pred,
})

print("\nFirst 10 predictions:")
print(df.head(10).to_string(index=False))

# Show some errors
errors = df[df["gold"] != df["pred"]]
print(f"\nNumber of errors: {len(errors)}")
print("Some error examples:")
print(errors.head(20).to_string(index=False))



Overall accuracy: 0.8392

Classification report:
              precision    recall  f1-score   support

    negative       0.83      0.95      0.89       116
     neutral       0.89      0.84      0.86       577
    positive       0.75      0.80      0.77       277

    accuracy                           0.84       970
   macro avg       0.82      0.86      0.84       970
weighted avg       0.84      0.84      0.84       970


First 10 predictions:
                                                                                                                                                                                                                                                                                 text     gold     pred
                                                                                           The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functio